In [1]:
import lightgbm as lgb
import numpy as np
import pandas as pd
import random
from load_data import load_data_with_folds, get_fold_data
from sklearn.metrics import mean_squared_error

X, y, patient_ids, fold_indices = load_data_with_folds(
    file_path="../data/slice_localization_data.csv",
    target_col="reference",
    drop_cols=["patientId"],
    n_folds=10,
    random_state=42
)

params = {
    'objective': 'regression',
    'boosting_type': 'dart',
    'num_iterations': 3,  # ensemble_size
    'num_leaves': 100,
    'learning_rate': 0.2,
    'feature_fraction_bynode': 0.2,
    'drop_rate': 0.2,
    'metric': 'mse',
    'verbose': 1,
    'max_depth': 10
}

In [2]:
X_train, y_train, X_test, y_test = get_fold_data(
    X, y, patient_ids, fold_indices, 0
)

In [3]:
def evaluate_fold(booster, X_train, y_train, X_test, y_test, fold_idx):
    y_pred_train = booster.predict(X_train)
    y_pred_test = booster.predict(X_test)
    train_mse = mean_squared_error(y_train, y_pred_train)
    test_mse = mean_squared_error(y_test, y_pred_test)
    return {
        'fold_idx': fold_idx,
        'train_mse': train_mse,
        'test_mse': test_mse,
        'train_samples': len(X_train),
        'test_samples': len(X_test)
    }

def print_cv_summary(fold_results):
    print("\n" + "=" * 60)
    print("CROSS-VALIDATION SUMMARY")
    print("=" * 60)
    for result in fold_results:
        print(f"Fold {result['fold_idx']:2d}: Train MSE={result['train_mse']:.4f}, Test MSE={result['test_mse']:.4f} (Train: {result['train_samples']:5d}, Test: {result['test_samples']:5d})")
    avg_test_mse = np.mean([r['test_mse'] for r in fold_results])
    std_test_mse = np.std([r['test_mse'] for r in fold_results])
    print(f"\nAverage Test MSE: {avg_test_mse:.4f} ± {std_test_mse:.4f}")

In [4]:
train_data = lgb.Dataset(X_train, label=y_train)
test_data = lgb.Dataset(X_test, label=y_test, reference=train_data)

booster = lgb.Booster(params, train_set=train_data)
booster.add_valid(test_data, "valid")

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.031336 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 78507
[LightGBM] [Info] Number of data points in the train set: 47838, number of used features: 378


In [5]:
for i in range(25):
    booster.update()  # add one more tree

    # Make a snapshot *after* this update
    model_str = booster.model_to_string()
    snap_booster = lgb.Booster(model_str=model_str)

    # Use the snapshot for SHAP – it will not be updated further
    shap_values = snap_booster.predict(X_train, pred_contrib=True)[:, :-1]

[LightGBM] [Info] Start training from score 47.366796
[LightGBM] [Info] DART iteration 0: No trees dropped
[LightGBM] [Info] DART iteration 1: No trees dropped
[LightGBM] [Info] DART iteration 2: No trees dropped
[LightGBM] [Info] DART iteration 3: No trees dropped
[LightGBM] [Info] DART iteration 4: No trees dropped
[LightGBM] [Info] DART iteration 5: No trees dropped
[LightGBM] [Info] DART iteration 6: No trees dropped
[LightGBM] [Info] DART iteration 7: Drop indices: [1, 5]
[LightGBM] [Info] DART iteration 8: No trees dropped
[LightGBM] [Info] DART iteration 9: No trees dropped
[LightGBM] [Info] DART iteration 10: Drop indices: [5]
[LightGBM] [Info] DART iteration 11: Drop indices: [4, 6, 9, 10]
[LightGBM] [Info] DART iteration 12: No trees dropped
[LightGBM] [Info] DART iteration 13: Drop indices: [1, 3, 8, 12]
[LightGBM] [Info] DART iteration 14: No trees dropped
[LightGBM] [Info] DART iteration 15: No trees dropped
[LightGBM] [Info] DART iteration 16: No trees dropped
[LightGBM] 